In [0]:
pip install azure-eventhub faker

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
from faker import Faker
import json
import time
import random

# Initialize faker
fake = Faker()


# Event Hub connection
CONNECTION_STR = "masked"
EVENT_HUB_NAME = "sitealert"

producer = EventHubProducerClient.from_connection_string(
    conn_str=CONNECTION_STR,
    eventhub_name=EVENT_HUB_NAME
)

projects = ["P1", "P2", "P3", "P4"]  # simulate sites
devices = ["D1", "D2", "D3", "D4", "D5"]

while True:
    project_id = random.choice(projects)
    device_id = random.choice(devices)

    event = {
        "event_id": fake.uuid4(),
        "project_id": project_id,
        "device_id": device_id,
        "timestamp": fake.iso8601(),
        "event_type": "progress_update",
        "value": random.randint(1, 100)
    }

    event_json = json.dumps(event)

    # 🔥 Partition key logic
    partition_key = f"{project_id}_{device_id}"

    event_data_batch = producer.create_batch(partition_key=partition_key)
    event_data_batch.add(EventData(event_json))

    producer.send_batch(event_data_batch)

    print(f"Sent: {event_json}")

    time.sleep(10)